In [1]:
import pandas as pd
import numpy as np
import country_converter as coco

Defining scoring funtions that calculate the risk for each feature category

In [2]:
def score_geopolitical(row):
    score = 0
    pol = row["political_stability_index"]
    if pol >= 1.0:    score += 0
    elif pol >= 0.0:  score += 20
    elif pol >= -1.0: score += 50
    else:             score += 100

    conflict = row["conflict_score"]
    if conflict < 20:   score += 0
    elif conflict < 40: score += 25
    elif conflict < 60: score += 50
    elif conflict < 80: score += 75
    else:               score += 100

    score += 100 * row["sanctions_exposure"]

    cpi = row["corruption_index"]
    if cpi >= 70:    score += 0
    elif cpi >= 50:  score += 20
    elif cpi >= 35:  score += 50
    elif cpi >= 20:  score += 75
    else:            score += 100

    score += 60 * row["export_restriction_risk"]

    dip = row["diplomatic_relations_score"]
    if dip >= 75:    score += 0
    elif dip >= 50:  score += 20
    elif dip >= 25:  score += 60
    else:            score += 100

    return min(score / 6, 100)


def score_geographic(row):
    score = 0
    dist = row["distance_km"]
    if dist < 500:     score += 0
    elif dist < 2000:  score += 20
    elif dist < 5000:  score += 50
    elif dist < 10000: score += 75
    else:              score += 100

    score += 40 * row["is_landlocked"]

    infra = row["infrastructure_score"]
    if infra >= 75:    score += 0
    elif infra >= 50:  score += 25
    elif infra >= 30:  score += 60
    else:              score += 100

    nd = row["natural_disaster_frequency"]
    if nd <= 1:   score += 0
    elif nd <= 3: score += 25
    elif nd <= 6: score += 60
    else:         score += 100


    return min(score / 4, 100)


def score_relationship(row):

    score = 0

    tenure = row["supplier_tenure_years"]
    if tenure >= 10:   score += 0
    elif tenure >= 5:  score += 20
    elif tenure >= 2:  score += 50
    else:              score += 80

    score += 60 if row["contract_type"] == "spot" else 0

    resp = row["communication_responsiveness_hours"]
    if resp <= 4:    score += 0
    elif resp <= 24: score += 25
    elif resp <= 72: score += 60
    else:            score += 100

    alts = row["num_alternative_suppliers"]
    if alts >= 3:   score += 0
    elif alts == 2: score += 25
    elif alts == 1: score += 60
    else:           score += 100

    tier_risk = {"Tier 1": 0, "Tier 2": 40, "Tier 3": 80}

    score += tier_risk.get(row["supplier_tier"], 50)

    return min(score / 5, 100)


def score_compliance(row):
    score = 0
    qi = row["quality_incidents"]
    if qi == 0:   score += 0
    elif qi <= 2: score += 30
    elif qi <= 5: score += 65
    else:         score += 100

    audit = row["audit_score"]
    if audit >= 85:   score += 0
    elif audit >= 70: score += 25
    elif audit >= 55: score += 60
    else:             score += 100

    score += 0 if row["iso_certified"] else 50

    viol = row["regulatory_violations"]
    if viol == 0:   score += 0
    elif viol <= 2: score += 40
    elif viol <= 5: score += 75
    else:           score += 100

    return min(score / 4, 100)


def score_financial(row):
    score = 0
    fsi = row["financial_stability_index"]
    if fsi >= 75:    score += 0
    elif fsi >= 55:  score += 25
    elif fsi >= 35:  score += 60
    else:            score += 100

    de = row["debt_to_equity"]
    if de < 0.5:   score += 0
    elif de < 1.0: score += 20
    elif de < 2.0: score += 50
    elif de < 3.0: score += 75
    else:          score += 100

    pd_ = row["payment_delay_days"]
    if pd_ <= 0:    score += 0
    elif pd_ <= 5:  score += 20
    elif pd_ <= 15: score += 55
    elif pd_ <= 30: score += 80
    else:           score += 100

    dep = row["dependency_ratio"]
    if dep < 20:   score += 0
    elif dep < 40: score += 25
    elif dep < 60: score += 55
    elif dep < 80: score += 80
    else:          score += 100

    rev = row["revenue_trend_yoy"]
    if rev >= 10:    score += 0
    elif rev >= 0:   score += 20
    elif rev >= -10: score += 55
    else:            score += 100


  # Larger companies are more financially stable and resilient to disruptions.
    emp = row["num_employees"]
    if emp >= 1000:  score += 0
    elif emp >= 200: score += 20
    elif emp >= 50:  score += 50
    else:            score += 80

    return min(score / 6, 100)


def score_operational(row):

    score = 0

    otd = row["on_time_delivery_rate"]
    if otd >= 95:    score += 0
    elif otd >= 85:  score += 20
    elif otd >= 70:  score += 55
    elif otd >= 50:  score += 80
    else:            score += 100

    #  how slow they are
    lt = row["avg_lead_time_days"]
    if lt <= 7:    score += 0
    elif lt <= 14: score += 20
    elif lt <= 30: score += 50
    elif lt <= 60: score += 75
    else:          score += 100

    # how unpredictable they are
    ltv = row["lead_time_variability"]
    if ltv <= 1:   score += 0
    elif ltv <= 3: score += 25
    elif ltv <= 7: score += 60
    else:          score += 100

    ofr = row["order_fulfillment_rate"]
    if ofr >= 97:    score += 0
    elif ofr >= 90:  score += 20
    elif ofr >= 80:  score += 55
    else:            score += 100

    pcu = row["capacity_utilization"]
    if pcu <= 75:    score += 0
    elif pcu <= 85:  score += 25
    elif pcu <= 95:  score += 60
    else:            score += 100

    dis = row["past_disruptions_lagged"]
    if dis == 0:   score += 0
    elif dis <= 1: score += 30
    elif dis <= 3: score += 65
    else:          score += 100

    rec = row["avg_recovery_time_lagged"]
    if rec <= 3:    score += 0
    elif rec <= 7:  score += 25
    elif rec <= 14: score += 55
    elif rec <= 30: score += 80
    else:           score += 100

    buf = row["inventory_buffer_days"]
    if buf >= 30:    score += 0
    elif buf >= 21:  score += 10
    elif buf >= 14:  score += 40
    elif buf >= 7:   score += 70
    else:            score += 100

    contracts = row["num_active_contracts"]
    if contracts <= 5:    score += 0
    elif contracts <= 15: score += 25
    elif contracts <= 30: score += 55
    else:                 score += 80

    transport_risk = {"air": 10, "land": 30, "sea": 60}
    score += transport_risk.get(row["transport_mode"], 40)

    strikes = row["labor_strike_history"]
    if strikes == 0:   score += 0
    elif strikes == 1: score += 40
    elif strikes == 2: score += 70
    else:              score += 100

    return min(score / 11, 100)

Combining category scores into a single weighted composite score

In [3]:
WEIGHTS = {
    "operational":  0.25,
    "financial":    0.20,
    "geopolitical": 0.25,
    "compliance":   0.15,
    "relationship": 0.10,
    "geographic":   0.10,
}

def composite_score(row):
    return (
        score_operational(row)  * WEIGHTS["operational"]  +
        score_financial(row)    * WEIGHTS["financial"]     +
        score_geopolitical(row) * WEIGHTS["geopolitical"]  +
        score_compliance(row)   * WEIGHTS["compliance"]    +
        score_relationship(row) * WEIGHTS["relationship"]  +
        score_geographic(row)   * WEIGHTS["geographic"]
    )

Converting composite score into a risk label (with override rules)

In [4]:
def classify_supplier(row):
    score = composite_score(row)    

    # Auto-Critical — conditions so severe the score doesn't matter
    if row["sanctions_exposure"] == 1:         return 4 
    if row["financial_stability_index"] < 15:  return 4 
    if row["regulatory_violations"] > 5:       return 4  
    if row["on_time_delivery_rate"] < 50:      return 4  

    # Convert composite score to label
    composite_label = (
        0 if score <= 20 else   
        1 if score <= 40 else   
        2 if score <= 60 else   
        3 if score <= 80 else   
        4                       
    )

    # Auto-High — bad enough to guarantee at least High, but not necessarily Critical
    if row["audit_score"] < 40:                return max(composite_label, 3)  
    if row["num_alternative_suppliers"] == 0:  return max(composite_label, 3)  
    if row["labor_strike_history"] >= 3:       return max(composite_label, 3)
    if row["export_restriction_risk"] == 1:    return max(composite_label, 3)  

    return composite_label

Agreggating suppliers scores into a one single supply chain risk score

In [5]:
def aggregate_supply_chain_risk(df):
    tier_weights = {"Tier 1": 3, "Tier 2": 2, "Tier 3": 1}
    df["tier_weight"] = df["supplier_tier"].map(tier_weights)

    weighted_score = (
        (df["risk_level"] * df["tier_weight"]).sum() /
        df["tier_weight"].sum()
    )

    tier1_critical = (
        (df["supplier_tier"] == "Tier 1") & (df["risk_level"] == 4)
    ).any()

    if tier1_critical:        overall = 4
    elif weighted_score <= 0.5: overall = 0
    elif weighted_score <= 1.5: overall = 1
    elif weighted_score <= 2.5: overall = 2
    elif weighted_score <= 3.5: overall = 3
    else:                       overall = 4

    labels = {0: "None", 1: "Low", 2: "Medium", 3: "High", 4: "Critical"}
    return {"weighted_score": round(weighted_score, 3),
            "overall_risk_level": overall,
            "overall_risk_label": labels[overall]}

Testing the scoring engine on one hardcoded supplier

In [6]:
test_supplier = {
    "political_stability_index": -0.5, "conflict_score": 50,
    "sanctions_exposure": 0, "corruption_index": 35,
    "export_restriction_risk": 1, "diplomatic_relations_score": 40,
    "distance_km": 7000, "is_landlocked": 1,
    "infrastructure_score": 40, "natural_disaster_frequency": 5,
    "supplier_region": "South Asia", "supplier_tenure_years": 3,
    "contract_type": "spot", "communication_responsiveness_hours": 48,
    "num_alternative_suppliers": 1, "supplier_tier": "Tier 1",
    "quality_incidents": 3, "audit_score": 62,
    "iso_certified": False, "regulatory_violations": 2,
    "financial_stability_index": 45, "debt_to_equity": 2.1,
    "payment_delay_days": 12, "dependency_ratio": 55,
    "revenue_trend_yoy": -5, "num_employees": 150,
    "on_time_delivery_rate": 76, "avg_lead_time_days": 28,
    "lead_time_variability": 5, "order_fulfillment_rate": 84,
    "capacity_utilization": 88, "past_disruptions_lagged": 2,
    "avg_recovery_time_lagged": 12, "inventory_buffer_days": 10,
    "num_active_contracts": 12, "transport_mode": "sea",
    "labor_strike_history": 1,
}

labels = {0: "None", 1: "Low", 2: "Medium", 3: "High", 4: "Critical"}
score = composite_score(test_supplier)
label = classify_supplier(test_supplier)

print(f"Composite Score : {score:.2f}")
print(f"Risk Level      : {label} — {labels[label]}")
print(f"\nCategory Scores:")
print(f"  Geopolitical  : {score_geopolitical(test_supplier):.1f}")
print(f"  Geographic    : {score_geographic(test_supplier):.1f}")
print(f"  Relationship  : {score_relationship(test_supplier):.1f}")
print(f"  Compliance    : {score_compliance(test_supplier):.1f}")
print(f"  Financial     : {score_financial(test_supplier):.1f}")
print(f"  Operational   : {score_operational(test_supplier):.1f}")

Composite Score : 54.98
Risk Level      : 3 — High

Category Scores:
  Geopolitical  : 45.0
  Geographic    : 58.8
  Relationship  : 46.0
  Compliance    : 53.8
  Financial     : 58.3
  Operational   : 54.1


Loading datasets

In [7]:
RAW = "../data/raw/"

In [8]:
# 1. Political Stability (World Bank) — scale: -2.5 to +2.5 / dataset from 2023
wb = pd.read_csv(RAW + "API_PV.EST_DS2_en_csv_v2_177.csv", skiprows=4)
wb = wb[["Country Name", "Country Code", "2023"]].rename(columns={
    "Country Name": "country",
    "Country Code": "country_code",
    "2023": "political_stability_index"
})


# Taiwan is missing from the World Bank dataset - because of the political reasons. Normally such a small country would be dropped
# but given Taiwan's critical role in the global supply chain, especially in semiconductors, it's better to have it, based on estimation
# with political stability index of 0.65, which reflects its stable political environment compared to other countries in the region.

taiwan_row = pd.DataFrame([{
    "country": "Taiwan",
    "country_code": "TWN",
    "political_stability_index": 0.65
}])
wb = pd.concat([wb, taiwan_row], ignore_index=True)


print("Missing political_stability_index:", wb["political_stability_index"].isna().sum())
print("Missing country_code:", wb["country_code"].isna().sum())
print("Missing country:", wb["country"].isna().sum())

print(wb[wb["political_stability_index"].isna()]["country"].tolist()) # They're all regional aggregates and groupings, not actual countries 

wb = wb.dropna(subset=["political_stability_index"])


# This dataset country name will be used as a reference

Missing political_stability_index: 61
Missing country_code: 0
Missing country: 0
['Africa Eastern and Southern', 'Africa Western and Central', 'Arab World', 'Central Europe and the Baltics', 'Channel Islands', 'Caribbean small states', 'Curacao', 'East Asia & Pacific (excluding high income)', 'Early-demographic dividend', 'East Asia & Pacific', 'Europe & Central Asia (excluding high income)', 'Europe & Central Asia', 'Euro area', 'European Union', 'Fragile and conflict affected situations', 'Faroe Islands', 'Gibraltar', 'High income', 'Heavily indebted poor countries (HIPC)', 'IBRD only', 'IDA & IBRD total', 'IDA total', 'IDA blend', 'IDA only', 'Isle of Man', 'Not classified', 'Latin America & Caribbean (excluding high income)', 'Latin America & Caribbean', 'Least developed countries: UN classification', 'Low income', 'Lower middle income', 'Low & middle income', 'Late-demographic dividend', 'St. Martin (French part)', 'Middle East, North Africa, Afghanistan & Pakistan', 'Middle incom

In [9]:
# 2. Infrastructure — scale: 1-5, rescaling to 0-100 / dataset from 2023
lpi = pd.read_excel(RAW + "International_LPI_from_2007_to_2023_0.xlsx", sheet_name="2023")
lpi = lpi[["Economy", "Infrastructure Score"]].rename(columns={
    "Economy": "country",
    "Infrastructure Score": "infrastructure_score"
})
lpi["infrastructure_score"] = ((lpi["infrastructure_score"] - 1) / (5 - 1) * 100).round(1)
lpi["country_code"] = coco.convert(lpi["country"].tolist(), to="ISO3")
lpi = lpi[lpi["country_code"] != "not found"]

print("Missing infrastructure_score:", lpi["infrastructure_score"].isna().sum())
print("Countries:", len(lpi))
print("Missing country_code:", lpi["country_code"].isna().sum())

TC<rkiye not found in regex


Missing infrastructure_score: 0
Countries: 138
Missing country_code: 0


In [10]:
# 3. EM-DAT Natural Disasters — disasters per year / dataset from (2010-2023)
emdat = pd.read_excel(RAW + "public_emdat_custom_request_2026-03-12_7a856d87-742f-4f5f-ae14-90bce777e4d7.xlsx",
                      sheet_name="EM-DAT Data")
emdat = emdat[emdat["Start Year"].between(2010, 2023)]
disaster_counts = emdat.groupby("Country").size().reset_index(name="disaster_count")
disaster_counts["country_code"] = coco.convert(disaster_counts["Country"].tolist(), to="ISO3")
disaster_counts = disaster_counts[disaster_counts["country_code"] != "not found"]
print("Missing Country:", emdat["Country"].isna().sum())
print("Countries:", len(disaster_counts))

Canary Islands not found in regex


Missing Country: 0
Countries: 208


In [11]:
# 4. CPI Corruption
cpi = pd.read_csv(RAW + "CPI2025_clean.csv")
cpi["country_code"] = coco.convert(cpi["country"].tolist(), to="ISO3")
cpi = cpi[cpi["country_code"] != "not found"]
print("Missing corruption_index:", cpi["corruption_index"].isna().sum())
print("Countries:", len(cpi))

Missing corruption_index: 0
Countries: 182


In [12]:
# 5. UCDP Conflict Events
ged = pd.read_csv(RAW + "GEDEvent_v25_1.csv", low_memory=False)
ged = ged[ged["year"].between(2021, 2024)]
conflict_counts = ged.groupby("country").size().reset_index(name="conflict_events")
conflict_counts["country_code"] = coco.convert(conflict_counts["country"].tolist(), to="ISO3")
conflict_counts = conflict_counts[conflict_counts["country_code"] != "not found"]
conflict_country = conflict_counts.groupby("country_code")["conflict_events"].sum().reset_index()
conflict_country.columns = ["country_code", "conflict_score"]
mn, mx = conflict_country["conflict_score"].min(), conflict_country["conflict_score"].max()
conflict_country["conflict_score"] = ((conflict_country["conflict_score"] - mn) / (mx - mn) * 100).round(1)
print(conflict_country.head())
print("Countries:", len(conflict_country))

  country_code  conflict_score
0          AFG            18.8
1          AGO             0.1
2          ARE             0.0
3          ARM             0.1
4          AZE             0.1
Countries: 84


In [35]:
# Merging all into country_ref
country_ref = wb.merge(lpi[["country_code", "infrastructure_score"]], on="country_code", how="left")
country_ref = country_ref.merge(cpi[["country_code", "corruption_index"]], on="country_code", how="left")
country_ref = country_ref.merge(disaster_counts[["country_code", "disaster_count"]], on="country_code", how="left")
country_ref["natural_disaster_frequency"] = (country_ref["disaster_count"].fillna(0) / 13).round(1)
country_ref = country_ref.drop(columns=["disaster_count"])
country_ref = country_ref.merge(conflict_country, on="country_code", how="left")
country_ref["conflict_score"] = country_ref["conflict_score"].fillna(0)

In [36]:
countries = ["Poland", "Germany", "China", "Australia", "Mexico", "Brazil", "Ukraine"]
country_ref[country_ref["country"].isin(countries)]

,country,country_code,political_stability_index,infrastructure_score,corruption_index,natural_disaster_frequency,conflict_score
10,Australia,AUS,0.917057,77.5,76.0,4.0,0.0
26,Brazil,BRA,-0.408568,55.0,35.0,8.5,21.2
35,China,CHN,-0.513097,75.0,43.0,30.2,0.0
48,Germany,DEU,0.586989,82.5,77.0,2.3,0.0
119,Mexico,MEX,-0.630761,45.0,27.0,8.6,40.3
148,Poland,POL,0.559585,62.5,53.0,1.9,0.0
190,Ukraine,UKR,-1.429823,35.0,36.0,1.9,100.0


In [37]:
print(country_ref.isnull().sum()[country_ref.isnull().sum() > 0])
print("Total rows:", len(country_ref))

infrastructure_score    68
corruption_index        24
dtype: int64
Total rows: 206


In [38]:
print(country_ref[country_ref.isnull().any(axis=1)][["country", "country_code", "infrastructure_score", "corruption_index", "conflict_score"]].to_string())

                            country country_code  infrastructure_score  corruption_index  conflict_score
0                             Aruba          ABW                   NaN               NaN             0.0
4                           Andorra          AND                   NaN               NaN             0.0
8                    American Samoa          ASM                   NaN               NaN             0.0
9               Antigua and Barbuda          ATG                  42.5               NaN             0.0
12                       Azerbaijan          AZE                   NaN              30.0             0.1
13                          Burundi          BDI                   NaN              17.0             0.6
23                           Belize          BLZ                   NaN              36.0             0.0
24                          Bermuda          BMU                   NaN               NaN             0.0
27                         Barbados          BRB       

In [39]:
# Maping each country to your archetype regions
REGION_COUNTRIES = {
    "Western Europe":  ["Germany", "France", "Netherlands", "Belgium", "Sweden",
                        "Denmark", "Finland", "Norway", "Austria", "Switzerland",
                        "Portugal", "Spain", "Ireland", "Italy", "Greece",
                        "Luxembourg", "Iceland"],
    "North America":   ["United States", "Canada", "Mexico"],
    "Australia":       ["Australia", "New Zealand"],
    "Eastern Europe":  ["Poland", "Romania", "Hungary", "Bulgaria", "Ukraine",
                        "Serbia", "Croatia", "Slovak Republic", "Czechia", "Slovenia",
                        "Lithuania", "Latvia", "Estonia", "Belarus", "Moldova"],
    "East Asia":       ["China", "Japan", "Korea, Rep.", "Taiwan",
                        "Hong Kong SAR, China", "Macao SAR, China", "Mongolia"],
    "South America":   ["Brazil", "Argentina", "Colombia", "Chile", "Peru",
                        "Ecuador", "Bolivia", "Paraguay", "Uruguay", "Venezuela, RB"],
    "Southeast Asia":  ["Viet Nam", "Thailand", "Indonesia", "Malaysia",
                        "Philippines", "Cambodia", "Myanmar", "Singapore",
                        "Lao PDR", "Brunei Darussalam"],
    "Middle East":     ["Saudi Arabia", "United Arab Emirates", "Iran, Islamic Rep.",
                        "Iraq", "Turkiye", "Jordan", "Lebanon", "Kuwait",
                        "Qatar", "Oman", "Bahrain", "Syrian Arab Republic",
                        "Yemen, Rep.", "Israel"],
    "South Asia":      ["India", "Pakistan", "Bangladesh", "Sri Lanka", "Nepal",
                        "Afghanistan", "Maldives", "Bhutan"],
    "Africa":          ["Nigeria", "Ethiopia", "Kenya", "Ghana", "Angola",
                        "Tanzania", "South Africa", "Mozambique", "Cameroon",
                        "Cote d'Ivoire", "Congo, Dem. Rep.", "Sudan", "Algeria",
                        "Egypt, Arab Rep.", "Morocco", "Tunisia", "Senegal",
                        "Uganda", "Rwanda", "Zambia", "Zimbabwe"],
    "Central Asia":    ["Kazakhstan", "Uzbekistan", "Tajikistan",
                        "Turkmenistan", "Kyrgyz Republic"],
}

# Adding region column to country_ref
country_to_region = {c: r for r, countries in REGION_COUNTRIES.items() for c in countries}
country_ref["region"] = country_ref["country"].map(country_to_region)


for col in ["infrastructure_score", "corruption_index"]:
    regional_median = country_ref.groupby("region")[col].transform("median")
    country_ref[col] = country_ref[col].fillna(regional_median).fillna(country_ref[col].median())
    
# Keeping only countries with a known region
country_ref_mapped = country_ref[country_ref["region"].notna()].reset_index(drop=True)
print(f"Countries with region mapping: {len(country_ref_mapped)}")
print(country_ref_mapped.groupby("region").size())

Countries with region mapping: 112
region
Africa            21
Australia          2
Central Asia       5
East Asia          7
Eastern Europe    15
Middle East       14
North America      3
South America     10
South Asia         8
Southeast Asia    10
Western Europe    17
dtype: int64


In [18]:
print(country_ref_mapped.isnull().sum()[country_ref_mapped.isnull().sum() > 0])
print("Total rows:", len(country_ref_mapped))

# No NaN values left

Series([], dtype: int64)
Total rows: 109


In [19]:
np.random.seed(42)

ARCHETYPES = {
    0: {  # None
        "political_stability_index":              1.5,
        "conflict_score":                              10,
        "sanctions_exposure":                     0.02,
        "corruption_index":                       78,
        "export_restriction_risk":                0.03,
        "diplomatic_relations_score":             85,
        "distance_km":                            800,
        "is_landlocked":                          0.05,
        "infrastructure_score":                   82,
        "natural_disaster_frequency":             1.0,
        "supplier_tenure_years":                  12,
        "communication_responsiveness_hours":     3,
        "num_alternative_suppliers":              4,
        "quality_incidents":                      0.2,
        "audit_score":                            90,
        "iso_certified":                          0.95,
        "regulatory_violations":                  0.1,
        "financial_stability_index":              85,
        "debt_to_equity":                         0.3,
        "payment_delay_days":                     -1,
        "dependency_ratio":                       15,
        "revenue_trend_yoy":                      12,
        "num_employees":                          2000,
        "on_time_delivery_rate":                  97,
        "avg_lead_time_days":                     6,
        "lead_time_variability":                  0.8,
        "order_fulfillment_rate":                 98,
        "capacity_utilization":                   65,
        "past_disruptions_lagged":                0.1,
        "avg_recovery_time_lagged":               2,
        "inventory_buffer_days":                  35,
        "num_active_contracts":                   4,
        "labor_strike_history":                   0.05,
    },
    1: {  # Low
        "political_stability_index":              0.8,
        "conflict_score":                              25,
        "sanctions_exposure":                     0.05,
        "corruption_index":                       60,
        "export_restriction_risk":                0.08,
        "diplomatic_relations_score":             70,
        "distance_km":                            2500,
        "is_landlocked":                          0.15,
        "infrastructure_score":                   65,
        "natural_disaster_frequency":             2.5,
        "supplier_tenure_years":                  7,
        "communication_responsiveness_hours":     10,
        "num_alternative_suppliers":              3,
        "quality_incidents":                      1.0,
        "audit_score":                            78,
        "iso_certified":                          0.80,
        "regulatory_violations":                  0.5,
        "financial_stability_index":              68,
        "debt_to_equity":                         0.7,
        "payment_delay_days":                     3,
        "dependency_ratio":                       28,
        "revenue_trend_yoy":                      6,
        "num_employees":                          700,
        "on_time_delivery_rate":                  91,
        "avg_lead_time_days":                     12,
        "lead_time_variability":                  2.0,
        "order_fulfillment_rate":                 93,
        "capacity_utilization":                   74,
        "past_disruptions_lagged":                0.5,
        "avg_recovery_time_lagged":               5,
        "inventory_buffer_days":                  26,
        "num_active_contracts":                   8,
        "labor_strike_history":                   0.15,
    },
    2: {  # Medium
        "political_stability_index":              0.0,
        "conflict_score":                              50,
        "sanctions_exposure":                     0.10,
        "corruption_index":                       45,
        "export_restriction_risk":                0.15,
        "diplomatic_relations_score":             52,
        "distance_km":                            5500,
        "is_landlocked":                          0.30,
        "infrastructure_score":                   50,
        "natural_disaster_frequency":             4.5,
        "supplier_tenure_years":                  4,
        "communication_responsiveness_hours":     28,
        "num_alternative_suppliers":              2,
        "quality_incidents":                      2.5,
        "audit_score":                            65,
        "iso_certified":                          0.55,
        "regulatory_violations":                  1.5,
        "financial_stability_index":              50,
        "debt_to_equity":                         1.3,
        "payment_delay_days":                     10,
        "dependency_ratio":                       48,
        "revenue_trend_yoy":                      1,
        "num_employees":                          250,
        "on_time_delivery_rate":                  80,
        "avg_lead_time_days":                     22,
        "lead_time_variability":                  4.5,
        "order_fulfillment_rate":                 84,
        "capacity_utilization":                   83,
        "past_disruptions_lagged":                1.5,
        "avg_recovery_time_lagged":               10,
        "inventory_buffer_days":                  18,
        "num_active_contracts":                   14,
        "labor_strike_history":                   0.50,
    },
    3: {  # High
        "political_stability_index":             -0.8,
        "conflict_score":                              75,
        "sanctions_exposure":                     0.20,
        "corruption_index":                       28,
        "export_restriction_risk":                0.35,
        "diplomatic_relations_score":             30,
        "distance_km":                            9000,
        "is_landlocked":                          0.55,
        "infrastructure_score":                   32,
        "natural_disaster_frequency":             6.5,
        "supplier_tenure_years":                  2,
        "communication_responsiveness_hours":     60,
        "num_alternative_suppliers":              1,
        "quality_incidents":                      4.5,
        "audit_score":                            52,
        "iso_certified":                          0.25,
        "regulatory_violations":                  3.0,
        "financial_stability_index":              32,
        "debt_to_equity":                         2.3,
        "payment_delay_days":                     22,
        "dependency_ratio":                       68,
        "revenue_trend_yoy":                     -7,
        "num_employees":                          90,
        "on_time_delivery_rate":                  68,
        "avg_lead_time_days":                     42,
        "lead_time_variability":                  7.5,
        "order_fulfillment_rate":                 74,
        "capacity_utilization":                   91,
        "past_disruptions_lagged":                3.0,
        "avg_recovery_time_lagged":               22,
        "inventory_buffer_days":                  8,
        "num_active_contracts":                   22,
        "labor_strike_history":                   1.20,
    },
    4: {  # Critical
        "political_stability_index":             -1.8,
        "conflict_score":                              95,
        "sanctions_exposure":                     0.60,
        "corruption_index":                       15,
        "export_restriction_risk":                0.65,
        "diplomatic_relations_score":             12,
        "distance_km":                            13000,
        "is_landlocked":                          0.75,
        "infrastructure_score":                   18,
        "natural_disaster_frequency":             8.5,
        "supplier_tenure_years":                  1,
        "communication_responsiveness_hours":     100,
        "num_alternative_suppliers":              0.3,
        "quality_incidents":                      7.0,
        "audit_score":                            38,
        "iso_certified":                          0.08,
        "regulatory_violations":                  5.5,
        "financial_stability_index":              14,
        "debt_to_equity":                         3.8,
        "payment_delay_days":                     40,
        "dependency_ratio":                       82,
        "revenue_trend_yoy":                     -18,
        "num_employees":                          35,
        "on_time_delivery_rate":                  52,
        "avg_lead_time_days":                     65,
        "lead_time_variability":                  12.0,
        "order_fulfillment_rate":                 61,
        "capacity_utilization":                   96,
        "past_disruptions_lagged":                5.5,
        "avg_recovery_time_lagged":               38,
        "inventory_buffer_days":                  4,
        "num_active_contracts":                   30,
        "labor_strike_history":                   2.50,
    },
}

NOISE = {
    "political_stability_index":          0.4,
    "conflict_score":                          8,
    "corruption_index":                   10,
    "diplomatic_relations_score":         12,
    "distance_km":                        800,
    "infrastructure_score":               10,
    "natural_disaster_frequency":         1.2,
    "supplier_tenure_years":              2,
    "communication_responsiveness_hours": 15,
    "num_alternative_suppliers":          0.8,
    "quality_incidents":                  1.0,
    "audit_score":                        8,
    "regulatory_violations":              0.8,
    "financial_stability_index":          10,
    "debt_to_equity":                     0.4,
    "payment_delay_days":                 5,
    "dependency_ratio":                   10,
    "revenue_trend_yoy":                  5,
    "num_employees":                      300,
    "on_time_delivery_rate":              5,
    "avg_lead_time_days":                 6,
    "lead_time_variability":              1.5,
    "order_fulfillment_rate":             5,
    "capacity_utilization":               6,
    "past_disruptions_lagged":            0.8,
    "avg_recovery_time_lagged":           5,
    "inventory_buffer_days":              5,
    "num_active_contracts":               4,
    "labor_strike_history":               0.5,
}

REGIONS = {
    0: ["Western Europe", "North America", "Australia"],
    1: ["Western Europe", "Eastern Europe", "East Asia"],
    2: ["East Asia", "Southeast Asia", "Eastern Europe", "South America"],
    3: ["South Asia", "Southeast Asia", "Middle East", "Africa"],
    4: ["Africa", "Central Asia", "Middle East", "South Asia"],
}

TIERS = {
    0: (["Tier 1", "Tier 2", "Tier 3"], [0.5, 0.4, 0.1]),
    1: (["Tier 1", "Tier 2", "Tier 3"], [0.4, 0.4, 0.2]),
    2: (["Tier 1", "Tier 2", "Tier 3"], [0.3, 0.4, 0.3]),
    3: (["Tier 1", "Tier 2", "Tier 3"], [0.2, 0.4, 0.4]),
    4: (["Tier 1", "Tier 2", "Tier 3"], [0.1, 0.3, 0.6]),
}

CONTRACTS = {
    0: (["long-term", "spot"], [0.90, 0.10]),
    1: (["long-term", "spot"], [0.75, 0.25]),
    2: (["long-term", "spot"], [0.55, 0.45]),
    3: (["long-term", "spot"], [0.35, 0.65]),
    4: (["long-term", "spot"], [0.15, 0.85]),
}

TRANSPORT = {
    0: (["air", "land", "sea"], [0.3, 0.5, 0.2]),
    1: (["air", "land", "sea"], [0.2, 0.5, 0.3]),
    2: (["air", "land", "sea"], [0.2, 0.3, 0.5]),
    3: (["air", "land", "sea"], [0.1, 0.3, 0.6]),
    4: (["air", "land", "sea"], [0.1, 0.2, 0.7]),
}

In [20]:
def generate_supplier(archetype_id):
    a = ARCHETYPES[archetype_id]

    def sample(key, low=None, high=None):
        val = np.random.normal(a[key], NOISE.get(key, 1))
        if low is not None:  val = max(val, low)
        if high is not None: val = min(val, high)
        return val

    row = {}

    # Picking a real country from the archetype's regions
    region = np.random.choice(REGIONS[archetype_id])
    region_countries = country_ref_mapped[country_ref_mapped["region"] == region]

    if len(region_countries) > 0:
        country_row = region_countries.sample(1).iloc[0]
        row["supplier_country"]            = country_row["country"]
        row["supplier_region"]             = region
        row["political_stability_index"]   = round(country_row["political_stability_index"], 2)
        row["infrastructure_score"]        = round(country_row["infrastructure_score"], 1)
        row["natural_disaster_frequency"]  = round(country_row["natural_disaster_frequency"], 1)
        row["corruption_index"]            = round(country_row["corruption_index"], 1)
        row["conflict_score"]                   = round(country_row["conflict_score"], 1)
    else:
        row["supplier_country"]            = "Unknown"
        row["supplier_region"]             = region
        row["political_stability_index"]   = round(sample("political_stability_index", -2.5, 2.5), 2)
        row["infrastructure_score"]        = round(sample("infrastructure_score", 0, 100), 1)
        row["natural_disaster_frequency"]  = round(sample("natural_disaster_frequency", 0, 10), 1)
        row["corruption_index"]            = round(sample("corruption_index", 0, 100), 1)
        row["conflict_score"]                   = round(sample("conflict_score", 0, 100), 1)

    # Geopolitical (remaining synthetic)
    row["sanctions_exposure"]                 = int(np.random.binomial(1, min(a["sanctions_exposure"], 1)))
    row["export_restriction_risk"]            = int(np.random.binomial(1, min(a["export_restriction_risk"], 1)))
    row["diplomatic_relations_score"]         = round(sample("diplomatic_relations_score", 0, 100), 1)

    # Geographic
    row["distance_km"]                        = round(sample("distance_km", 100, 20000))
    row["is_landlocked"]                      = int(np.random.binomial(1, a["is_landlocked"]))

    # Relationship
    row["supplier_tenure_years"]              = max(0, round(sample("supplier_tenure_years", 0, 30)))
    row["contract_type"]                      = np.random.choice(CONTRACTS[archetype_id][0], p=CONTRACTS[archetype_id][1])
    row["communication_responsiveness_hours"] = round(sample("communication_responsiveness_hours", 1, 200), 1)
    row["num_alternative_suppliers"]          = max(0, round(sample("num_alternative_suppliers", 0, 10)))
    row["supplier_tier"]                      = np.random.choice(TIERS[archetype_id][0], p=TIERS[archetype_id][1])

    # Compliance
    row["quality_incidents"]                  = max(0, int(np.random.poisson(a["quality_incidents"])))
    row["audit_score"]                        = round(sample("audit_score", 0, 100), 1)
    row["iso_certified"]                      = int(np.random.binomial(1, a["iso_certified"]))
    row["regulatory_violations"]              = max(0, int(np.random.poisson(a["regulatory_violations"])))

    # Financial
    row["financial_stability_index"]          = round(sample("financial_stability_index", 0, 100), 1)
    row["debt_to_equity"]                     = round(sample("debt_to_equity", 0, 10), 2)
    row["payment_delay_days"]                 = round(sample("payment_delay_days", -5, 90), 1)
    row["dependency_ratio"]                   = round(sample("dependency_ratio", 0, 100), 1)
    row["revenue_trend_yoy"]                  = round(sample("revenue_trend_yoy", -50, 50), 1)
    row["num_employees"]                      = max(5, round(sample("num_employees", 5, 50000)))

    # Operational
    row["on_time_delivery_rate"]              = round(sample("on_time_delivery_rate", 0, 100), 1)
    row["avg_lead_time_days"]                 = max(1, round(sample("avg_lead_time_days", 1, 120)))
    row["lead_time_variability"]              = round(sample("lead_time_variability", 0, 30), 1)
    row["order_fulfillment_rate"]             = round(sample("order_fulfillment_rate", 0, 100), 1)
    row["capacity_utilization"]               = round(sample("capacity_utilization", 10, 100), 1)
    row["past_disruptions_lagged"]            = max(0, int(np.random.poisson(a["past_disruptions_lagged"])))
    row["avg_recovery_time_lagged"]           = round(sample("avg_recovery_time_lagged", 0, 90), 1)
    row["inventory_buffer_days"]              = max(0, round(sample("inventory_buffer_days", 0, 60)))
    row["num_active_contracts"]               = max(1, round(sample("num_active_contracts", 1, 50)))
    row["transport_mode"]                     = np.random.choice(TRANSPORT[archetype_id][0], p=TRANSPORT[archetype_id][1])
    row["labor_strike_history"]               = max(0, int(np.random.poisson(a["labor_strike_history"])))

    # Enforcing correlations
    if row["financial_stability_index"] < 35:
        row["payment_delay_days"] += np.random.uniform(5, 20)
    if row["audit_score"] < 55:
        row["regulatory_violations"] += np.random.randint(0, 3)
    if row["capacity_utilization"] > 88:
        row["lead_time_variability"] += np.random.uniform(1, 4)

    return row

In [21]:
def generate_dataset(n_per_class=200):
    rows = []
    archetype_ids = []

    for archetype_id in range(5):
        for _ in range(n_per_class):
            rows.append(generate_supplier(archetype_id))
            archetype_ids.append(archetype_id)

    df = pd.DataFrame(rows)
    df["archetype"]  = archetype_ids
    df["risk_level"] = df.apply(classify_supplier, axis=1)

    return df

In [22]:
df = generate_dataset(n_per_class=200)

labels = {0: "None", 1: "Low", 2: "Medium", 3: "High", 4: "Critical"}

print(f"Dataset shape: {df.shape}")
print(f"\nClass distribution:")
counts = df["risk_level"].value_counts().sort_index()
for k, v in counts.items():
    print(k, labels[k], v)

df.to_csv("../data/suppliers.csv", index=False)
print("\nSaved → ../data/suppliers.csv")

Dataset shape: (1000, 40)

Class distribution:
0 None 292
1 Low 161
2 Medium 75
3 High 166
4 Critical 306

Saved → ../data/suppliers.csv
